# Chapter 2 — GPT Text Decoder from Scratch

**Goal**: Build a GPT-style autoregressive language model.
This is the text "backbone" of our VLM.

## Key difference from ViT (Chapter 1)

| | ViT (Vision Encoder) | GPT (Language Decoder) |
|---|---|---|
| Attention | Bidirectional (all→all) | Causal (past→current only) |
| Goal | Encode image to features | Predict next token |
| Loss | Contrastive (Ch 3) | Cross-entropy |

## Architecture
```
Token IDs → Embedding + Positional Embedding
                ↓
         N × [CausalAttention + FFN]   (with residual + LayerNorm)
                ↓
           LayerNorm
                ↓
         LM Head (Linear, weight-tied with embedding)
                ↓
        Logits (vocab_size) → softmax → next token probabilities
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 2.1 Causal Attention Mask

The most important difference between ViT and GPT attention.

During training, we feed the entire sequence at once and predict every
next token in parallel. But token *i* must not see token *i+1* (or later),
otherwise it would just "copy" the answer.

We enforce this by masking the upper triangle of the attention matrix.

In [ ]:
T = 6  # sequence length for illustration

# The causal mask: True means BLOCK attention
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ViT: all ones (full attention)
axes[0].imshow(torch.ones(T, T), cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('ViT: Bidirectional Attention\n(every token sees every token)')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

# GPT: lower triangle only
allowed = (~mask).float()
axes[1].imshow(allowed, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('GPT: Causal Attention\n(token i only sees tokens 0..i)')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')

for ax in axes:
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))

plt.tight_layout()
plt.savefig('figures/ch02_attention_masks.png', dpi=100)
plt.show()

## 2.2 Building the GPT Model

In [ ]:
from multimodal_from_scratch.language.gpt import GPT, GPTConfig

# Small config for testing on CPU
cfg = GPTConfig(
    vocab_size=1000,
    context_len=128,
    embed_dim=256,
    depth=4,
    num_heads=4,
    dropout=0.0,
)

gpt = GPT(cfg)
print(f"GPT parameters: {gpt.count_parameters():,}")

# Forward pass
input_ids = torch.randint(0, cfg.vocab_size, (2, 32))   # batch=2, seq=32
logits = gpt(input_ids)

print(f"\nInput:  {input_ids.shape}   (B, T)")
print(f"Logits: {logits.shape}  (B, T, vocab_size)")

## 2.3 Language Modelling Loss

For each position *t*, the model predicts token *t+1*.
We shift inputs by 1 to create (input, target) pairs:

```
Input:   [The, cat, sat, on, the]
Target:  [cat, sat, on, the, mat]
```

In [ ]:
# Compute cross-entropy loss
input_ids = torch.randint(0, cfg.vocab_size, (2, 16))
logits = gpt(input_ids)   # (B, T, V)

# Shift: predict token i+1 from position i
# logits[:, :-1, :] are predictions for positions 0..T-2
# input_ids[:, 1:]  are targets  at positions 1..T-1
loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, cfg.vocab_size),
    input_ids[:, 1:].reshape(-1)
)

# Random model baseline: loss ≈ log(vocab_size)
import math
print(f"Loss (untrained): {loss.item():.3f}")
print(f"Expected random:  {math.log(cfg.vocab_size):.3f}  (= ln({cfg.vocab_size}))")

## 2.4 Text Generation

In [ ]:
# Generate text from an untrained model (will be random, but shows the pipeline)
prompt = torch.tensor([[1, 2, 3]])   # pretend these are token ids

generated = gpt.generate(
    input_ids=prompt,
    max_new_tokens=10,
    temperature=1.0,
    top_k=50,
)

print(f"Prompt tokens:    {prompt.tolist()[0]}")
print(f"Generated tokens: {generated.tolist()[0]}")  
print(f"Shape: {generated.shape}  (1, prompt_len + new_tokens)")

## 2.5 Weight Tying

GPT shares weights between the input embedding and the output LM head.
This reduces parameters and empirically improves performance.

```python
self.lm_head.weight = self.token_embed.weight  # ← same tensor object
```

Intuition: the same "meaning" of a token should be consistent whether
it's used as input or predicted as output.

In [ ]:
# Verify weight tying
print("LM head weight IS token embedding weight:",
      gpt.lm_head.weight is gpt.token_embed.weight)

# Without tying, how many extra params would we have?
extra = cfg.vocab_size * cfg.embed_dim
print(f"\nWeight tying saves {extra:,} parameters  "
      f"({cfg.vocab_size} × {cfg.embed_dim})")

## Summary

We now have a GPT decoder that:
1. Embeds tokens + positions
2. Applies causal (masked) self-attention — no future peeking
3. Uses a feed-forward network per position
4. Predicts the next token with a weight-tied LM head
5. Can generate text autoregressively

Notice the `visual_prefix` argument in `GPT.forward()` — in **Chapter 4**
we will fill this with visual tokens from ViT, turning GPT into a VLM.

**Next**: Chapter 3 — CLIP Contrastive Learning